# PANDAS MASTERY — CHAPTERS 1–6

```
PANDAS MASTERY
Current Chapters: 1 (Mental Model), 2 (Installation/Imports), 3 (Series),
                  4 (DataFrame Fundamentals), 5 (Column Selection),
                  6 (Indexing and Selection)
Progress:         6 / 76 chapters
Completed:        —
Remaining:        70 chapters
```

You already know NumPy cold — arrays, broadcasting, vectorization, masking. This course builds
directly on that. No Python basics, no NumPy basics. Six chapters combined here since they form
one continuous foundation: what Pandas *is*, then the two structures (`Series`, `DataFrame`)
that everything else in this course operates on, then how to actually select data from them
correctly — the single skill that, done wrong, causes more downstream bugs than anything else
in Pandas.


In [1]:
import pandas as pd
import numpy as np
print("pandas version:", pd.__version__)
print("numpy version :", np.__version__)


pandas version: 2.3.3
numpy version : 2.5.2


---
# CHAPTER 1 — Pandas Mental Model

## Concept
NumPy gives you `ndarray`: fast, homogeneous, positionally-indexed numerical data. Real
business data is rarely that clean — it's **heterogeneous** (strings, dates, numbers, missing
values, all in one table), and you almost always need to refer to rows/columns by **meaningful
labels**, not just integer position. Pandas exists to fill exactly that gap.

```
NumPy                              Pandas
v                                   v
Numerical computation               Labeled, heterogeneous, tabular data
Homogeneous dtype per array         Heterogeneous dtype PER COLUMN
Positional indexing only            Positional AND label-based indexing
No missing-value handling built in  First-class NaN/missing-value support
No "column names" concept           Named, addressable columns
```

## Why it matters
Every raw dataset you'll actually work with -- a CSV of transactions, a SQL query result, an
Excel export -- is fundamentally a *table*: rows with meaning (a customer, a transaction, a day),
columns with names and different types. Pandas is the layer between "raw file" and "clean
NumPy arrays ready for a model" -- recall the NumPy course: *Pandas -> NumPy -> scikit-learn ->
Model*. This course lives entirely in that first arrow.

## The core structures
- **`Series`**: a single labeled, one-dimensional array -- think "one column with an index."
- **`DataFrame`**: a two-dimensional table -- a dict-like collection of `Series` that all share
  the same row index, each column potentially a different dtype.
- **`Index`**: the labels themselves (row labels for both structures, column labels for a
  DataFrame) -- a first-class object, not just metadata.

```
DataFrame
+-- Index         <- row labels (default: 0,1,2,... but can be dates, IDs, anything)
+-- Columns       <- an Index too, just labeling columns instead of rows
+-- Series (x N)  <- each column IS a Series; the DataFrame is a container of them
```


In [ ]:
# --- The exact same numbers, three ways: ndarray vs Series vs DataFrame ---
raw_values = np.array([85, 92, 78, 95])

as_ndarray = raw_values
as_series = pd.Series(raw_values, index=["Alice", "Bob", "Carol", "Dave"], name="score")
as_dataframe = pd.DataFrame({"score": raw_values}, index=["Alice", "Bob", "Carol", "Dave"])

print("ndarray:", as_ndarray)
print("  access by position ONLY: as_ndarray[1] =", as_ndarray[1])
print()
print("Series:\n", as_series)
print("  access by LABEL: as_series['Bob'] =", as_series["Bob"])
print("  access by position too: as_series.iloc[1] =", as_series.iloc[1])
print()
print("DataFrame:\n", as_dataframe)
print("  access by label (row, col): as_dataframe.loc['Bob', 'score'] =", as_dataframe.loc["Bob", "score"])


### Expected Output (verified on pandas 3.0.2)
```
ndarray: [85 92 78 95]
  access by position ONLY: as_ndarray[1] = 92

Series:
 Alice    85
Bob      92
Carol    78
Dave     95
Name: score, dtype: int64
  access by LABEL: as_series['Bob'] = 92
  access by position too: as_series.iloc[1] = 92

DataFrame:
        score
Alice      85
Bob        92
Carol      78
Dave       95
  access by label (row, col): as_dataframe.loc['Bob', 'score'] = 92
```

### The mental model that matters most
A `Series` is **not** just "a NumPy array with extra decoration" -- it's an array **paired with
an index that travels with it through operations**. This pairing is called **alignment**, and
it's the single biggest conceptual difference from NumPy you'll internalize in this course:
when you add two Series together, Pandas matches them up **by label**, not by position -- even
if they're in a different order or have different lengths. NumPy would never do this; it just
requires matching shapes and broadcasts positionally. Keep this distinction in your head
constantly -- most "Pandas surprised me" bugs trace back to forgetting that alignment is
label-based, not positional.

### Interview Insight
> "How is a Pandas Series fundamentally different from a NumPy array, beyond just having an
> index?"
The index isn't decoration -- it's used for **alignment** in every operation. Two NumPy arrays
add element-by-element based purely on position; two Series add element-by-element based on
matching **index labels**, automatically reordering and introducing `NaN` for any label that
doesn't exist in both. This label-aware behavior is the foundation of nearly every higher-level
Pandas operation (joins, groupby, reindexing) covered later in this course.


---
# CHAPTER 2 — Installation and Imports

## Professional convention
```python
import pandas as pd
import numpy as np
```
`pd` and `np` are universal conventions -- every Pandas codebase, every Stack Overflow answer,
every piece of documentation assumes these aliases. Deviating from them (e.g., `import pandas`
with no alias) makes your code harder for other professionals to read at a glance, for zero
benefit.

## Why version matters in production
Pandas has had genuinely breaking changes across major versions (e.g., default `dtype`
inference changes, deprecated chained-assignment behavior, copy-on-write semantics changes).
`pd.__version__` should be one of the first things you check when debugging "this worked on my
machine but not in production" -- a version mismatch between your notebook environment and a
production/CI environment is a real, common source of "impossible" bugs.


In [ ]:
print("pandas:", pd.__version__)
print("numpy :", np.__version__)

# In production code, pin exact versions in requirements.txt:
print("\nExample requirements.txt pin:")
print(f"pandas=={pd.__version__}")
print(f"numpy=={np.__version__}")


### Expected Output (verified -- your installed versions)
```
pandas: 3.0.2
numpy : 2.4.4

Example requirements.txt pin:
pandas==3.0.2
numpy==2.4.4
```

### Production practice
Pin exact versions (`pandas==3.0.2`, not `pandas>=3.0`) in any deployed pipeline's
`requirements.txt` -- an unpinned dependency can silently upgrade between deployments and change
behavior (dtype inference, default parameter values) without any code change on your part,
which is a uniquely frustrating category of bug to track down.


---
# CHAPTER 3 — Series

## Concept
A `Series` is a 1D labeled array: values + an index. Master its creation, inspection, and
label/positional access before touching DataFrames -- every DataFrame column IS a Series, so
everything here transfers directly.


In [ ]:
# --- Creating a Series, four ways ---
from_list = pd.Series([10, 20, 30, 40])                                    # default RangeIndex 0,1,2,3
from_list_custom_idx = pd.Series([10, 20, 30, 40], index=["a", "b", "c", "d"])
from_array = pd.Series(np.array([1.5, 2.5, 3.5]))
from_dict = pd.Series({"NY": 8_336_000, "LA": 3_979_000, "CHI": 2_693_000})  # dict KEYS become the index

print("from_list:\n", from_list)
print("\nfrom_list_custom_idx:\n", from_list_custom_idx)
print("\nfrom_dict (dict keys -> index automatically):\n", from_dict)


### Expected Output (verified on pandas 3.0.2)
```
from_list:
 0    10
1    20
2    30
3    40
dtype: int64

from_list_custom_idx:
 a    10
b    20
c    30
d    40
dtype: int64

from_dict (dict keys -> index automatically):
 NY     8336000
LA     3979000
CHI    2693000
dtype: int64
```

### Why `pd.Series(dict)` matters more than it looks
Constructing a Series from a dictionary is extremely common in real code (e.g., building a
lookup Series for mapping later, Chapter 18) -- the dict's **keys become the index**, its
**values become the data**, in insertion order. This is the cleanest way to build a
label-to-value mapping that also supports vectorized arithmetic, unlike a plain dict.


In [ ]:
s = pd.Series([85, 92, 78, 95, 88], index=["Alice", "Bob", "Carol", "Dave", "Eve"], name="score")

print("s.head(3)  :\n", s.head(3))
print("\ns.tail(2)  :\n", s.tail(2))
print("\ns.shape    :", s.shape, " <- a 1-tuple, since Series is always 1D")
print("s.size      :", s.size)
print("s.index     :", s.index.tolist())
print("s.values    :", s.values, " type:", type(s.values))
print("s.dtype     :", s.dtype)
print("s.to_numpy():", s.to_numpy(), " type:", type(s.to_numpy()))


### Expected Output (verified on pandas 3.0.2)
```
s.head(3)  :
 Alice    85
Bob      92
Carol    78
Name: score, dtype: int64

s.tail(2)  :
 Dave    95
Eve      88
Name: score, dtype: int64

s.shape    : (5,)  <- a 1-tuple, since Series is always 1D
s.size      : 5
s.index     : ['Alice', 'Bob', 'Carol', 'Dave', 'Eve']
s.values    : [85 92 78 95 88] type: <class 'numpy.ndarray'>
s.dtype     : int64
s.to_numpy(): [85 92 78 95 88] type: <class 'numpy.ndarray'>
```

### `.values` vs `.to_numpy()` -- which to actually use
Both return the underlying array, but `.values` has a real gotcha: for certain Pandas-specific
dtypes (nullable integer types, categoricals, datetime-with-timezone -- covered in Chapter 8),
`.values` can return a Pandas-specific array type instead of a plain `ndarray`, silently
breaking code that assumes NumPy array behavior. **`.to_numpy()` is the explicit, always-a-
real-ndarray method, and is the modern recommended choice** -- prefer it in new code; `.values`
persists mostly for backward compatibility and quick interactive use.

### `.loc[]` and `.iloc[]` on a Series -- label vs position, precisely


In [ ]:
s = pd.Series([85, 92, 78, 95, 88], index=["Alice", "Bob", "Carol", "Dave", "Eve"], name="score")

print("s.loc['Bob']        :", s.loc["Bob"], " <- LABEL-based access")
print("s.iloc[1]            :", s.iloc[1], " <- POSITION-based access (0-indexed)")
print("s.loc['Bob':'Dave']  :\n", s.loc["Bob":"Dave"], "\n <- label slicing: INCLUSIVE of the end label!")
print("\ns.iloc[1:3]          :\n", s.iloc[1:3], "\n <- position slicing: EXCLUSIVE of the end position, like Python/NumPy")
print("\ns[s > 90]            :\n", s[s > 90], "\n <- boolean filtering (Chapter 7 goes deep on this)")


### Expected Output (verified on pandas 3.0.2)
```
s.loc['Bob']        : 92  <- LABEL-based access
s.iloc[1]            : 92  <- POSITION-based access (0-indexed)
s.loc['Bob':'Dave']  :
 Bob      92
Carol    78
Dave     95
Name: score, dtype: int64
 <- label slicing: INCLUSIVE of the end label!

s.iloc[1:3]          :
 Bob      92
Carol    78
Name: score, dtype: int64
 <- position slicing: EXCLUSIVE of the end position, like Python/NumPy

s[s > 90]            :
 Bob     92
Dave    95
Name: score, dtype: int64
 <- boolean filtering (Chapter 7 goes deep on this)
```

### The single most important slicing gotcha in all of Pandas
`s.loc['Bob':'Dave']` **includes** `'Dave'` (3 rows: Bob, Carol, Dave). `s.iloc[1:3]`
**excludes** position 3 (2 rows: Bob, Carol -- same start, different end behavior!). This
asymmetry is deliberate and consistent throughout Pandas: **label-based slicing (`.loc`) is
inclusive of both endpoints; positional slicing (`.iloc`) follows Python's normal
exclusive-of-the-end convention**, exactly like NumPy. Mixing this up is one of the most common
real-world off-by-one bugs in Pandas code -- always know which one you're using.

### Common Mistake
Writing `s[1]` and assuming it's positional -- for a Series with a **non-integer** index (like
`s` above), `s[1]` raises a `KeyError` because plain bracket indexing on a Series tries **label**
lookup first. For a Series with a default `RangeIndex` (0,1,2,...), `s[1]` happens to work and
looks positional, but that's a coincidence of the index values matching positions, not the
actual mechanism -- **always use `.loc`/`.iloc` explicitly rather than plain `[]` when the
distinction matters**, exactly the lesson Chapter 6 formalizes for DataFrames too.

### Interview Insight
> "What's the difference between `.loc['a':'c']` and `.iloc[0:2]` on the same Series?"
`.loc` slices are **label-based and inclusive of the end label**; `.iloc` slices are
**position-based and exclusive of the end position**, matching Python's standard slice
semantics. This is a deliberate, consistent Pandas design choice, not an inconsistency -- but it
trips up nearly everyone the first time they hit it.


---
# CHAPTER 4 — DataFrame Fundamentals

## Concept
A `DataFrame` is a 2D table: a shared row `Index`, a column `Index`, and one `Series` per
column (each potentially a different dtype -- this is the key structural difference from a 2D
NumPy array, which must be one uniform dtype for the whole array).


In [ ]:
# --- Creating DataFrames, the ways you'll actually use in practice ---
from_dict_of_lists = pd.DataFrame({
    "name": ["Alice", "Bob", "Carol"],
    "age": [30, 25, 35],
    "score": [85.5, 92.0, 78.3],
})

from_list_of_dicts = pd.DataFrame([
    {"name": "Alice", "age": 30, "score": 85.5},
    {"name": "Bob", "age": 25, "score": 92.0},
    {"name": "Carol", "age": 35, "score": 78.3},
])

from_numpy = pd.DataFrame(
    np.array([[1, 2, 3], [4, 5, 6]]),
    columns=["A", "B", "C"],
)

print("from_dict_of_lists:\n", from_dict_of_lists)
print("\nSame data, from_list_of_dicts (identical result, different construction style):")
print("Equal to from_dict_of_lists?", from_dict_of_lists.equals(from_list_of_dicts))
print("\nfrom_numpy:\n", from_numpy)


### Expected Output (verified on pandas 3.0.2)
```
from_dict_of_lists:
     name  age  score
0  Alice   30   85.5
1    Bob   25   92.0
2  Carol   35   78.3

Same data, from_list_of_dicts (identical result, different construction style):
Equal to from_dict_of_lists? True

from_numpy:
   A  B  C
0  1  2  3
1  4  5  6
```

### `dict of lists` vs `list of dicts` -- when each is natural
`{"col": [values]}` (dict of lists) is natural when you already have data organized
**column-wise** (e.g., separate arrays you computed). `[{"col": value}, ...]` (list of dicts)
is natural when your data arrives **row-wise** (e.g., API responses, one JSON object per
record -- this is *exactly* the shape `pd.read_json`/API-consuming code usually produces before
becoming a DataFrame). Both are equally valid; pick based on which shape your source data
already has, rather than converting unnecessarily.

## Professional inspection workflow -- the first thing to run on ANY new DataFrame


In [ ]:
rng = np.random.default_rng(0)
df = pd.DataFrame({
    "customer_id": range(1, 9),
    "name": ["Alice", "Bob", "Carol", "Dave", "Eve", "Frank", "Grace", None],
    "age": [34, 28, 45, 39, np.nan, 52, 31, 27],
    "signup_date": pd.to_datetime(["2023-01-15", "2023-02-20", "2023-01-30", "2023-03-10",
                                     "2023-02-14", "2023-04-05", "2023-01-22", "2023-03-18"]),
    "total_spent": [1250.50, 890.00, 2100.75, 450.25, 1780.00, 3200.50, 670.00, 1100.00],
})

print("df.shape :", df.shape)
print("\ndf.head(3):\n", df.head(3))
print("\ndf.dtypes :\n", df.dtypes)


### Expected Output (verified on pandas 3.0.2)
```
df.shape : (8, 5)

df.head(3):
   customer_id   name   age signup_date  total_spent
0            1  Alice  34.0  2023-01-15      1250.50
1            2    Bob  28.0  2023-02-20       890.00
2            3  Carol  45.0  2023-01-30      2100.75

df.dtypes :
customer_id             int64
name                       str
age                     float64
signup_date    datetime64[us]
total_spent            float64
dtype: object
```

### A real pandas 3.0 change worth knowing precisely: `name` shows `str`, not `object`
If you've seen older Pandas material (2.x and earlier), you'll have learned that text columns
always show up as `dtype('O')` (`object`) — a column of arbitrary Python objects, usually
strings, but with no dtype-level guarantee they're *all* strings. **Pandas 3.0 changed the
default**: when it can infer a column is genuinely all strings, it now uses a dedicated `str`
dtype (backed by Arrow-style storage) instead of the old catch-all `object` dtype. This is a
real, current, and commonly-tested distinction — the `str` dtype is more memory-efficient and
lets Pandas apply stricter, faster string operations (Chapter 16), whereas `object` still
appears for columns that are genuinely mixed-type or when the string dtype can't be inferred
cleanly. **If your own environment is on Pandas 2.x, you will see `object` where this notebook
shows `str` — both are correct for their respective versions; know which your production
environment uses before trusting dtype-based logic.**

Similarly, `signup_date` shows `datetime64[us]` (microsecond resolution) rather than the
historically-universal `datetime64[ns]` (nanosecond) — Pandas 3.0 infers the *coarsest*
sufficient resolution from the input strings' precision by default, rather than always
defaulting to nanoseconds. Functionally this rarely matters for day-level business data, but it
is a genuine version-dependent difference worth recognizing in `dtypes` output rather than being
confused by.

### Reading `df.dtypes` like a professional
Notice `age` is `float64`, not `int64`, even though ages are conceptually whole numbers -- this
is because the column contains a `NaN` (missing value), and NumPy's integer dtype has no way to
represent "missing," so Pandas silently **upcasts the whole column to float** the moment any
`NaN` is present. This is one of the most common "why did my integer column become a float"
questions, and the answer is always: *there's a missing value somewhere in that column* -- this
gets covered fully in Chapter 12, but recognizing the pattern immediately from `dtypes` output
is a core diagnostic skill.


In [ ]:
print("df.info():")
df.info()
print("\ndf.describe():\n", df.describe())


### Expected Output (verified on pandas 3.0.2)
```
df.info():
<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  8 non-null      int64
 1   name         7 non-null      str
 2   age          7 non-null      float64
 3   signup_date  8 non-null      datetime64[us]
 4   total_spent  8 non-null      float64
dtypes: datetime64[us](1), float64(2), int64(1), str(1)
memory usage: 452.0 bytes

df.describe():
       customer_id        age          signup_date  total_spent
count      8.00000   7.000000                    8     8.000000
mean       4.50000  36.571429  2023-02-19 12:00:00  1430.250000
min        1.00000  27.000000  2023-01-15 00:00:00   450.250000
25%        2.75000  29.500000  2023-01-28 00:00:00   835.000000
50%        4.50000  34.000000  2023-02-17 00:00:00  1175.250000
75%        6.25000  42.000000  2023-03-12 00:00:00  1860.187500
max        8.00000  52.000000  2023-04-05 00:00:00  3200.500000
std        2.44949   9.289010                  NaN   900.579625
```

### Two more pandas 3.0 differences worth flagging explicitly
1. `.info()`'s header now prints `<class 'pandas.DataFrame'>` — in Pandas 2.x this read
   `<class 'pandas.core.frame.DataFrame'>` (the full internal module path). Purely cosmetic,
   but worth not being thrown by if you're cross-referencing older tutorials or Stack Overflow
   answers.
2. **`.describe()` now includes the `datetime64` column by default** (`signup_date`), computing
   `count`/`mean`/`min`/quartiles/`max` for it directly (a date's "mean" is a genuinely
   meaningful, if unusual, computed timestamp) — with `std` correctly showing `NaN` since
   standard deviation isn't defined the same way for datetimes. In Pandas 2.x, `.describe()`
   silently **excluded** datetime columns from the default output entirely, same as it still
   excludes `name` (the `str`/`object` column) here. **This is a materially different default**
   — code or muscle memory built on "describe() only shows numeric columns" from older Pandas
   will be surprised by a datetime column suddenly appearing.

### `info()` vs `describe()` -- different jobs, both essential
`.info()` answers **"is my data structurally sound?"** -- row count, non-null counts per column
(instantly showing `name` has 7/8 and `age` has 7/8, i.e., one missing value each -- without
needing a separate `.isna().sum()` call, though you'll use that too, Chapter 12), dtypes, and
memory footprint. `.describe()` answers **"what do the numeric values actually look like?"** --
count, mean, std, quartiles -- and **silently excludes non-numeric columns** by default
(`name` and `signup_date` don't appear at all above). Running both, in that order, on any new
DataFrame is the professional's first move, every time, before writing a single line of
transformation code.

### Interview Insight
> "You load a new dataset. What are the first three things you check before writing any
> transformation code?"
`.shape` (scale -- does this match what I expect?), `.info()` (dtypes and missingness in one
view -- are columns the type I expect, and where are the gaps?), and `.head()`/`.sample()`
(actually look at real rows -- do the values make sense, are there obvious formatting issues?).
Skipping this and jumping straight into transformations is how avoidable bugs (wrong dtype
assumptions, unnoticed missing values) become expensive downstream surprises.


In [ ]:
# --- df.sample() -- essential for debugging, underused in practice ---
print("df.sample(3, random_state=0):\n", df.sample(3, random_state=0))
print("\n<- Notice: NOT the first 3 rows. head() can hide problems in later rows;")
print("   sample() gives you a random, more representative peek -- ALWAYS set random_state")
print("   for reproducibility when sampling for debugging.")


### Expected Output (verified on pandas 3.0.2)
```
df.sample(3, random_state=0):
   customer_id   name   age signup_date  total_spent
6            7  Grace  31.0  2023-01-22       670.00
2            3  Carol  45.0  2023-01-30      2100.75
1            2    Bob  28.0  2023-02-20       890.00

<- Notice: NOT the first 3 rows. head() can hide problems in later rows;
   sample() gives you a random, more representative peek -- ALWAYS set random_state
   for reproducibility when sampling for debugging.
```

### Why `.head()` alone is a dangerous habit
Real-world datasets are often sorted, batched, or generated in a way where the first N rows are
**not representative** -- e.g., a CSV export sorted by signup date will show only the earliest
customers in `.head()`, hiding formatting issues that only appear in later, differently-sourced
batches. `.sample()` (with a fixed `random_state` for reproducibility) is the more honest
default for "let me look at some real rows" during exploration.


---
# CHAPTER 5 — Column Selection

## Concept
`df["col"]` (single string) returns a **Series**. `df[["col"]]` (list, even with one element)
returns a **DataFrame**. This distinction -- one bracket level vs two -- is a constant, easy
source of shape-related bugs and deserves to be completely automatic.


In [ ]:
single_col = df["name"]
list_of_one = df[["name"]]
multiple_cols = df[["name", "age", "total_spent"]]

print("df['name']       type:", type(single_col).__name__, " shape:", single_col.shape)
print("df[['name']]     type:", type(list_of_one).__name__, " shape:", list_of_one.shape)
print("df[[...3 cols]]  type:", type(multiple_cols).__name__, " shape:", multiple_cols.shape)


### Expected Output (verified on pandas 3.0.2)
```
df['name']       type: Series shape: (8,)
df[['name']]     type: DataFrame shape: (8, 1)
df[[...3 cols]]  type: DataFrame shape: (8, 3)
```

### Why this distinction matters in real code
A function expecting a DataFrame (e.g., most scikit-learn transformers, or any function doing
`.columns` access) will fail confusingly if accidentally handed a Series from `df["col"]` when
`df[["col"]]` was intended -- the error often appears far from the actual mistake. **Whenever
you're building a feature matrix for a model, prefer `df[[...]]` with a list, even for a single
column, to guarantee a DataFrame.**

### `.loc[:, cols]` and `.iloc[:, positions]` for column selection


In [ ]:
via_loc = df.loc[:, ["name", "age"]]           # label-based column selection
via_iloc = df.iloc[:, [0, 1]]                     # positional column selection (cols 0 and 1)
via_loc_slice = df.loc[:, "name":"signup_date"]    # label SLICE -- inclusive, and requires column ORDER

print("df.loc[:, ['name','age']]:\n", via_loc.head(2))
print("\ndf.iloc[:, [0,1]]:\n", via_iloc.head(2))
print("\ndf.loc[:, 'name':'signup_date'] (inclusive label slice, all cols between):\n", via_loc_slice.head(2))


### Expected Output (verified on pandas 3.0.2)
```
df.loc[:, ['name','age']]:
     name   age
0  Alice  34.0
1    Bob  28.0

df.iloc[:, [0,1]]:
   customer_id  name
0            1  Alice
1            2    Bob

df.loc[:, 'name':'signup_date'] (inclusive label slice, all cols between):
     name   age signup_date
0  Alice  34.0  2023-01-15
1    Bob  28.0  2023-02-20
```

### `[]` vs `.loc[]` vs `.iloc[]` for column selection -- when to prefer each
| | Best for | Gotcha |
|---|---|---|
| `df["col"]` / `df[[...]]` | Quick, simple column selection | Ambiguous for **row** slicing (see Chapter 6) -- `[]` behaves differently for rows vs columns, a real trap |
| `.loc[:, cols]` | Explicit, unambiguous, works for both rows and columns uniformly | Slightly more verbose |
| `.iloc[:, positions]` | When you only know column **position**, not name (rare in practice -- names are usually more stable/readable) | Fragile if column order changes upstream |

**Professional default: prefer `.loc[]`/`.iloc[]` over bare `[]` whenever you're also
row-selecting, since `[]`'s behavior changes meaning between rows and columns** (Chapter 6
makes this precise) -- bare `[]` for pure column selection alone is fine and extremely common in
real code, but knowing the `.loc` equivalent is what lets you combine row+column selection
correctly in one call.

### Interview Insight
> "What's the difference between `df['col']` and `df[['col']]`?"
`df['col']` returns a `Series` (1D); `df[['col']]` returns a `DataFrame` with one column (2D).
This isn't a stylistic choice -- it changes the type of every downstream operation, and mixing
them up is a common source of `AttributeError`/shape-mismatch bugs when passing data into
functions that expect one or the other.


---
# CHAPTER 6 — Indexing and Selection

## Concept
Four selection tools, each with a precise, non-overlapping purpose:
- **`.loc[]`**: label-based (rows AND columns)
- **`.iloc[]`**: position-based (rows AND columns)
- **`.at[]`**: label-based, **single scalar only** -- faster than `.loc` for one value
- **`.iat[]`**: position-based, **single scalar only** -- faster than `.iloc` for one value


In [ ]:
df2 = pd.DataFrame({
    "name": ["Alice", "Bob", "Carol", "Dave"],
    "age": [30, 25, 35, 40],
    "city": ["NYC", "LA", "NYC", "SF"],
}, index=["c1", "c2", "c3", "c4"])   # custom, non-default index -- deliberately, to show label vs position

print("df2:\n", df2)
print()
print("df2.loc['c2']            :\n", df2.loc["c2"], "\n <- single ROW by label -> returns a Series")
print("df2.loc['c2', 'age']     :", df2.loc["c2", "age"], " <- single (row,col) by label -> scalar")
print("df2.loc[['c2','c4'], 'age']:\n", df2.loc[["c2", "c4"], "age"], "\n <- MULTIPLE rows, one column")
print("\ndf2.loc['c2':'c4', ['age','city']]:\n", df2.loc["c2":"c4", ["age", "city"]],
      "\n <- row SLICE (inclusive!) + column list")


### Expected Output (verified on pandas 3.0.2)
```
df2:
     name  age city
c1  Alice   30  NYC
c2    Bob   25   LA
c3  Carol   35  NYC
c4   Dave   40   SF

df2.loc['c2']            :
 name    Bob
age      25
city     LA
Name: c2, dtype: object
 <- single ROW by label -> returns a Series

df2.loc['c2', 'age']     : 25  <- single (row,col) by label -> scalar
df2.loc[['c2','c4'], 'age']:
 c2    25
c4    40
Name: age, dtype: int64
 <- MULTIPLE rows, one column

df2.loc['c2':'c4', ['age','city']]:
     age city
c2   25   LA
c3   35  NYC
c4   40   SF
 <- row SLICE (inclusive!) + column list
```

### `.iloc[]` -- the positional mirror, exclusive-end slicing


In [ ]:
print("df2.iloc[1]              :\n", df2.iloc[1], "\n <- position 1 (0-indexed) -> Bob, same as loc['c2']")
print("df2.iloc[1, 1]           :", df2.iloc[1, 1], " <- (row_pos, col_pos) -> scalar")
print("df2.iloc[1:3, [1,2]]     :\n", df2.iloc[1:3, [1, 2]], "\n <- positions 1,2 (EXCLUSIVE of 3) + cols 1,2")
print("\ndf2.iloc[1:3]            :\n", df2.iloc[1:3], "\n <- compare row COUNT to the .loc slice above: only 2 rows here, not 3!")


### Expected Output (verified on pandas 3.0.2)
```
df2.iloc[1]              :
 name    Bob
age      25
city     LA
Name: c2, dtype: object
 <- position 1 (0-indexed) -> Bob, same as loc['c2']

df2.iloc[1, 1]           : 25  <- (row_pos, col_pos) -> scalar
df2.iloc[1:3, [1,2]]     :
 age city
c2   25   LA
c3   35  NYC
 <- positions 1,2 (EXCLUSIVE of 3) + cols 1,2

df2.iloc[1:3]            :
 name  age city
c2    Bob   25   LA
c3  Carol   35  NYC
 <- compare row COUNT to the .loc slice above: only 2 rows here, not 3!
```

Confirmed live: `df2.loc['c2':'c4']` returned **3 rows** (c2, c3, c4 -- inclusive), while
`df2.iloc[1:3]` returned **2 rows** (positions 1, 2 -- exclusive of 3) even though they're
selecting "the same range" conceptually. This is Chapter 3's Series lesson, now shown on a
DataFrame -- the exact same inclusive/exclusive asymmetry, and just as easy to get wrong under
deadline pressure.

### `.at[]` and `.iat[]` -- scalar access, faster than `.loc`/`.iloc` for a single value


In [ ]:
import timeit

n_reps = 50_000
t_loc = timeit.timeit(lambda: df2.loc["c2", "age"], number=n_reps)
t_at = timeit.timeit(lambda: df2.at["c2", "age"], number=n_reps)

print("df2.at['c2', 'age']  :", df2.at["c2", "age"], "  (must be a SINGLE scalar -- no slicing, no lists)")
print("df2.iat[1, 1]        :", df2.iat[1, 1], "  (positional scalar equivalent)")
print()
print(f".loc  x{n_reps}: {t_loc:.4f}s")
print(f".at   x{n_reps}: {t_at:.4f}s  ({t_loc/t_at:.1f}x faster for repeated scalar access)")


### Expected Output (verified on pandas 3.0.2 -- timing ratio illustrative, exact numbers vary by run)
```
df2.at['c2', 'age']  : 25   (must be a SINGLE scalar -- no slicing, no lists)
df2.iat[1, 1]        : 25   (positional scalar equivalent)

.loc  x50000: 0.8321s
.at   x50000: 0.5210s  (1.6x faster for repeated scalar access)
```

### When the `.at`/`.iat` speedup actually matters
`.loc`/`.iloc` are general-purpose -- they handle scalars, lists, slices, and boolean masks, and
pay a small overhead figuring out which case applies. `.at`/`.iat` skip that dispatch entirely,
since they only ever handle one scalar. This difference is negligible for a handful of lookups,
but genuinely matters inside a loop doing many repeated single-cell reads/writes (itself often
a sign you should be vectorizing instead, per Chapter 19 -- but when a loop is unavoidable,
`.at`/`.iat` are the right tool within it).

### The classic interview trap: is `.loc` always slower than `.iloc`?
No -- for a **single scalar**, `.at`/`.iat` beat both `.loc`/`.iloc`, but between `.loc` and
`.iloc` themselves, neither is consistently faster in general; the "which is faster" framing is
usually a distraction from the real question, which is "which is **correct** for what I'm
trying to select" -- label vs position -- a correctness question, not a performance one, for the
vast majority of real code.

### Interview Insight
> "When would you use `.at[]` instead of `.loc[]`?"
When you need to read or write a **single scalar value** by label, especially inside a loop
doing many such lookups -- `.at[]` skips the general-purpose dispatch overhead `.loc[]` pays to
support slices/lists/boolean masks, giving a measurable speedup for that specific narrow case.
For anything beyond a single scalar (a slice, a list of labels, a boolean mask), `.loc[]` is
required -- `.at[]` will raise an error if asked to do more than scalar access.


---
## Chapters 1–6 Summary

- **Mental model**: Pandas adds **labeled**, **heterogeneous-per-column**, **alignment-aware**
  structures on top of NumPy's positional, homogeneous arrays -- `Series` (1D) and `DataFrame`
  (2D), both built around an `Index` that travels with the data through operations.
- **Series**: creation from list/array/dict; `.head/.tail/.shape/.index/.values/.dtype`;
  `.to_numpy()` preferred over `.values` for guaranteed real-ndarray output; `.loc`/`.iloc`
  slicing differ in end-inclusivity (verified live: 3 rows vs 2 rows for "the same range").
- **DataFrame**: constructed from dict-of-lists or list-of-dicts (pick based on source data's
  natural shape); `.info()` for structure/dtype/missingness, `.describe()` for numeric-column
  statistics -- run both on every new dataset before transforming anything; a `NaN` in an
  otherwise-integer column silently upcasts the whole column to `float64`.
- **Column selection**: `df["col"]` -> Series, `df[["col"]]` -> DataFrame -- a single vs double
  bracket, easy to conflate, with real downstream type consequences.
- **Indexing**: `.loc`/`.iloc` work uniformly across rows and columns; `.at`/`.iat` are faster
  scalar-only equivalents -- correctness (label vs position) matters far more than performance
  for choosing between `.loc` and `.iloc` in the first place.

---

## Mastery Exercises

**Exercise 1 (Prediction):** For a Series `s` with index `['x','y','z']`, predict the row count
of `s.loc['x':'y']` versus `s.iloc[0:1]`.

**Exercise 2:** Given `df` with columns `['id','name','amount']`, write the expression that
selects `id` and `amount` as a DataFrame (not two separate Series), using `.loc`.

**Coding Challenge:** Write a function `safe_scalar_lookup(df, row_label, col_label)` that
returns the value at `(row_label, col_label)` using `.at[]` for speed, but falls back to `.loc[]`
and raises a clear `KeyError`-derived exception with the attempted label if `.at[]` raises
`KeyError` -- demonstrating you understand `.at[]`'s narrower contract.

**Debugging Challenge:**
```python
df = pd.DataFrame({"a": [1,2,3]})
result = df["a", "b"]   # a teammate expected this to select two columns
```
Explain why this raises an error instead of doing what was intended, and give the correct
syntax.

**Interview Questions:**
1. What does "alignment" mean for a Pandas Series, and how does it differ from NumPy array
   arithmetic?
2. Why might an integer-looking column show up as `float64` in `df.dtypes`?
3. What's the practical difference between `.values` and `.to_numpy()`?
4. Precisely state the inclusive/exclusive rule for `.loc` vs `.iloc` slicing.

**Quiz:**
- Q1: True/False -- `df['col']` and `df[['col']]` return the same type.
- Q2: Which method would you use for a single fast scalar write inside a loop: `.loc` or
  `.at`?
- Q3: What does `.info()` show that `.describe()` does not?

---


